# PINN — 1D Wave Equation

$$\frac{\partial^2 u}{\partial t^2} - \beta \frac{\partial^2 u}{\partial x^2} = 0, \quad x \in [0,1],\ t \in [0,1]$$

**IC (displacement):** $u(x,0) = \sin(\pi x) + \tfrac{1}{2}\sin(\beta\pi x)$  
**IC (velocity):** $\dfrac{\partial u}{\partial t}(x,0) = 0$  
**BC:** $u(0,t) = u(1,t) = 0$

**Exact solution** ($\beta = 3$):
$$u(x,t) = \sin(\pi x)\cos(2\pi t) + \tfrac{1}{2}\sin(3\pi x)\cos(6\pi t)$$

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pinns import PINN

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Problem Setup

In [ ]:
BETA = 3.0   # wave speed

def exact(x, t):
    return (np.sin(np.pi * x) * np.cos(2 * np.pi * t)
            + 0.5 * np.sin(BETA * np.pi * x) * np.cos(2 * BETA * np.pi * t))

def ic_displacement(x):
    """u(x, 0)"""
    return np.sin(np.pi * x) + 0.5 * np.sin(BETA * np.pi * x)

# Sanity: exact at t=0 should equal IC
x_check = np.linspace(0, 1, 200)
assert np.allclose(exact(x_check, 0), ic_displacement(x_check), atol=1e-6)
# Sanity: du/dt at t=0 should be 0
dt = 1e-5
dudt_approx = (exact(x_check, dt) - exact(x_check, 0)) / dt
assert np.allclose(dudt_approx, 0.0, atol=1e-3)
print("Sanity checks passed.")

## 2. Data Generation

In [ ]:
N_pde   = 5000
N_ic    = 400   # displacement IC
N_ic_ut = 400   # velocity IC
N_bc    = 400

# --- PDE collocation: random in [0,1] x [0,1] ---
x_pde = torch.rand(N_pde, 2, device=device)   # col0=x, col1=t

# --- IC displacement: t=0 ---
x_ic_np = np.random.uniform(0, 1, size=(N_ic, 1))
x_ic = torch.tensor(
    np.hstack([x_ic_np, np.zeros((N_ic, 1))]), dtype=torch.float32, device=device
)
u_ic = torch.tensor(ic_displacement(x_ic_np), dtype=torch.float32, device=device)

# --- IC velocity: du/dt(x,0) = 0 ---
x_ic_ut_np = np.random.uniform(0, 1, size=(N_ic_ut, 1))
x_ic_ut = torch.tensor(
    np.hstack([x_ic_ut_np, np.zeros((N_ic_ut, 1))]), dtype=torch.float32, device=device
)
ut_ic = torch.zeros(N_ic_ut, 1, device=device)   # target: 0

# --- BC: u(0,t)=0, u(1,t)=0 ---
t_bc_np = np.random.uniform(0, 1, size=(N_bc, 1))
x_left  = torch.tensor(np.hstack([np.zeros((N_bc//2, 1)), t_bc_np[:N_bc//2]]),
                        dtype=torch.float32, device=device)
x_right = torch.tensor(np.hstack([np.ones((N_bc//2, 1)),  t_bc_np[N_bc//2:]]),
                        dtype=torch.float32, device=device)
x_bc = torch.cat([x_left, x_right], dim=0)
u_bc = torch.zeros(N_bc, 1, device=device)

print(f"PDE      : {x_pde.shape}")
print(f"IC (u)   : {x_ic.shape}   target: {u_ic.shape}")
print(f"IC (u_t) : {x_ic_ut.shape}   target: {ut_ic.shape}  (all zeros)")
print(f"BC       : {x_bc.shape}   target: {u_bc.shape}")

## 3. PDE Residual

Wave equation: $u_{tt} - \beta\, u_{xx} = 0$  
Both `u_tt` and `u_xx` are computed with autograd.

In [ ]:
def wave_residual(model, xt):
    """
    PDE: u_tt - beta * u_xx = 0
    xt[:, 0] = x,  xt[:, 1] = t
    """
    u = model(xt)                                           # [N, 1]

    # First derivatives
    grads1 = torch.autograd.grad(
        u, xt,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]                                                    # [N, 2]: [u_x, u_t]

    u_x = grads1[:, 0:1]
    u_t = grads1[:, 1:2]

    # Second derivatives
    u_xx = torch.autograd.grad(
        u_x, xt,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True
    )[0][:, 0:1]

    u_tt = torch.autograd.grad(
        u_t, xt,
        grad_outputs=torch.ones_like(u_t),
        create_graph=True
    )[0][:, 1:2]

    return u_tt - BETA * u_xx

## 4. Model & Training

> **Note:** `x_ic_ut` / `ut_ic` 인자로 초기 속도 조건 `∂u/∂t(x,0)=0`을 추가로 강제합니다.

In [ ]:
model = PINN(
    layers=[2, 64, 64, 64, 64, 1],
    activation="tanh",
    residual=False,
    skip=False,
    use_fourier=False,
    use_ntk=True,
).to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
model.fit(
    pde_fn=wave_residual,
    x_pde=x_pde,
    x_bc=x_bc,
    u_bc=u_bc,
    x_ic=x_ic,
    u_ic=u_ic,
    x_ic_ut=x_ic_ut,     # <-- 초기 속도 조건
    ut_ic=ut_ic,          # <-- du/dt(x,0) = 0
    epochs=20000,
    lr=1e-3,
    w_pde=1.0,
    w_bc=10.0,
    w_ic=10.0,
    w_ic_ut=10.0,
    ntk_adaptive=True,
    ntk_update_every=1000,
    log_every=2000,
    save_path="./best_wave.pt",
)

## 5. Loss History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.semilogy(model.loss_history["total"],  label="Total",   lw=1.5)
ax.semilogy(model.loss_history["pde"],    label="PDE",     lw=1)
ax.semilogy(model.loss_history["bc"],     label="BC",      lw=1)
ax.semilogy(model.loss_history["ic"],     label="IC (u)",  lw=1)
ax.semilogy(model.loss_history["ic_ut"],  label="IC (u_t)", lw=1, linestyle="--")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Training Loss")
ax.legend()

ax = axes[1]
ax.plot(model.loss_history["w_bc"], label="w_bc", lw=1.5)
ax.plot(model.loss_history["w_ic"], label="w_ic", lw=1.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("Weight")
ax.set_title("NTK Adaptive Weights")
ax.legend()

plt.tight_layout()
plt.show()

## 6. Prediction vs Exact — Heatmap

In [ ]:
Nx, Nt = 200, 200
x_grid = np.linspace(0, 1, Nx)
t_grid = np.linspace(0, 1, Nt)
XX, TT = np.meshgrid(x_grid, t_grid)

U_exact = exact(XX, TT)

model.eval()
xt_test = torch.tensor(
    np.stack([XX.ravel(), TT.ravel()], axis=1), dtype=torch.float32, device=device
)
with torch.no_grad():
    U_pred = model(xt_test).cpu().numpy().reshape(Nt, Nx)

U_err = np.abs(U_pred - U_exact)
vmin, vmax = U_exact.min(), U_exact.max()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
kw = dict(origin="lower", extent=[0, 1, 0, 1], aspect="auto")

im0 = axes[0].imshow(U_exact, cmap="RdBu_r", vmin=vmin, vmax=vmax, **kw)
axes[0].set_title("Exact $u(x,t)$"); axes[0].set_xlabel("$x$"); axes[0].set_ylabel("$t$")
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(U_pred, cmap="RdBu_r", vmin=vmin, vmax=vmax, **kw)
axes[1].set_title("PINN $u(x,t)$"); axes[1].set_xlabel("$x$"); axes[1].set_ylabel("$t$")
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(U_err, cmap="hot", **kw)
axes[2].set_title("Absolute Error"); axes[2].set_xlabel("$x$"); axes[2].set_ylabel("$t$")
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()

rel_l2 = np.linalg.norm(U_pred - U_exact) / (np.linalg.norm(U_exact) + 1e-10)
print(f"Relative L2 error: {rel_l2:.4e}")

## 7. Slice Comparison at Fixed Times

In [ ]:
t_slices = [0.0, 0.1, 0.25, 0.5, 1.0]

fig, axes = plt.subplots(1, len(t_slices), figsize=(16, 3), sharey=True)
for ax, t_val in zip(axes, t_slices):
    t_idx = int(round(t_val * (Nt - 1)))
    ax.plot(x_grid, U_exact[t_idx], "k-",  lw=2,   label="Exact")
    ax.plot(x_grid, U_pred[t_idx],  "r--", lw=1.5, label="PINN")
    ax.set_title(f"$t = {t_val}$")
    ax.set_xlabel("$x$")
    ax.legend(fontsize=8)

axes[0].set_ylabel("$u$")
plt.suptitle("Solution Slices: Exact vs PINN", y=1.02)
plt.tight_layout()
plt.show()